# quantum-cortex - N1 control run (Kaggle)

Settings required: **Accelerator = GPU T4** (single GPU used in v0), **Internet = On**.
Expected wall time: ~1.5-2.5 h for 500M byte-tokens. Follow `docs/EXECUTION-2026-08-07.md` (E3-E5).

In [ ]:
!git clone https://github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install jsonschema datasets

In [ ]:
# Build the open data slice (FineWeb-Edu, ODC-By: attribution in OSS-SURVEY section 5)
import numpy as np, os
from datasets import load_dataset
budget = 600_000_000
os.makedirs('data', exist_ok=True)
out = np.memmap('data/fineweb_edu_bytes.bin', dtype=np.uint16, mode='w+', shape=(budget,))
ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
i = 0
for ex in ds:
    b = ex['text'].encode('utf-8', errors='ignore')
    n = min(len(b), budget - i)
    out[i:i+n] = np.frombuffer(b[:n], dtype=np.uint8)
    i += n
    if i >= budget: break
out.flush(); print(f'wrote {i:,} byte-tokens')

In [ ]:
import os, shutil
ck = 'runs/kaggle_t4/ckpt.pt'
os.makedirs('runs/kaggle_t4', exist_ok=True)
# If a previous version's output is attached as an Input, reuse its checkpoint (cross-session resume)
if not os.path.exists(ck):
    for root, _, files in os.walk('/kaggle/input'):
        if 'ckpt.pt' in files:
            shutil.copy(os.path.join(root, 'ckpt.pt'), ck)
            print('checkpoint recovered from', root)
            break
flag = '--resume' if os.path.exists(ck) else ''
# 645 min training budget: the version always finishes cleanly under Kaggle's 12 h wall,
# so outputs (checkpoint) publish even when the run is not done yet. Rerun -> auto-resume.
!python train.py --config configs/kaggle_t4.json {flag} --time-budget-min 645


In [ ]:
print(open('metrics/LATEST.md').read())
print(open('metrics/runs.jsonl').read())  # copy the record line -> commit per EXECUTION E5